# Imports

In [1]:
import os, gc, csv, time, json, pickle
import numpy as np
import pandas as pd
from datetime import datetime

from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
from torchvision import models, transforms

from transformers import (
    ViTModel, ViTImageProcessor,
    AutoModel, AutoImageProcessor,
    CLIPVisionModel, CLIPProcessor
)

import captum
from captum.attr import (
    IntegratedGradients, Saliency, Occlusion, GradientShap
)

import shap
import quantus
from quantus.metrics.faithfulness.infidelity import Infidelity
from quantus.metrics.robustness.max_sensitivity import MaxSensitivity
from quantus.metrics.complexity.sparseness import Sparseness
from quantus.metrics.faithfulness.sensitivity_n import SensitivityN

from quantus.metrics.faithfulness.infidelity import Infidelity
from quantus.metrics.faithfulness.sensitivity_n import SensitivityN
from quantus.metrics.faithfulness.faithfulness_correlation import FaithfulnessCorrelation

from quantus.metrics.faithfulness.faithfulness_correlation import FaithfulnessCorrelation
from quantus.metrics.faithfulness.sensitivity_n import SensitivityN

from quantus.metrics.complexity.sparseness import Sparseness

from quantus.metrics.robustness.avg_sensitivity import AvgSensitivity
from quantus.metrics.localisation.pointing_game import PointingGame
from quantus.metrics.complexity.complexity import Complexity




/home/aysel/tfe/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


# Config

In [2]:
CURRENT_DATASET = "Flickr8k"
BASE_DIR = "TFE_Data"
DATASETS_DIR = os.path.join(BASE_DIR, "Datasets")

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Using device:", device)


Using device: cuda


# Dataset Loader

In [3]:
class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        try:
            img = Image.open(path).convert("RGB")
        except:
            img = Image.new("RGB", (224, 224), (0, 0, 0))

        if self.transform:
            img = self.transform(img)

        return img


# Vision Models

## ResNet50

In [4]:
def get_resnet50_model(device):
    weights = models.ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=weights)
    model.fc = nn.Linear(2048, 1000)
    return model.to(device).eval(), weights.transforms()


## MobileNetV3

In [5]:
def get_mobilenet_v3_model(device):
    weights = models.MobileNet_V3_Large_Weights.DEFAULT
    model = models.mobilenet_v3_large(weights=weights)
    model.classifier = nn.Linear(960, 1000)
    return model.to(device).eval(), weights.transforms()


## ViT

In [6]:
class ViTWithHead(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.backbone = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
        self.head = nn.Linear(self.backbone.config.hidden_size, 1000)

    def forward(self, x):
        out = self.backbone(pixel_values=x)
        cls = out.last_hidden_state[:, 0]
        return self.head(cls)

def get_vit_model(device):
    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    return ViTWithHead(device).to(device).eval(), transform


## PVT

In [7]:
class CaptumWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        # HuggingFace models expect pixel_values=...
        outputs = self.model(pixel_values=x)
        return outputs.logits


In [8]:
class PVTWithHead(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("Zetatech/pvt-tiny-224").to(device).eval()

        # PVT uses embed_dims list, last stage = CLS dimension
        hidden = self.backbone.config.embed_dims[-1]

        self.head = nn.Linear(hidden, 1000).to(device)

    def forward(self, x):
        out = self.backbone(pixel_values=x)
        cls = out.last_hidden_state[:, 0, :]   # CLS token
        return self.head(cls)

from transformers import AutoModelForImageClassification, AutoImageProcessor

def get_pvt_model(device):
    processor = AutoImageProcessor.from_pretrained("Zetatech/pvt-tiny-224")
    base_model = AutoModelForImageClassification.from_pretrained(
        "Zetatech/pvt-tiny-224"
    ).to(device).eval()

    model = CaptumWrapper(base_model)

    def transform(img):
        return processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0)

    return model, transform




## Clip Vision

In [9]:
class CLIPWithHead(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.backbone = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")
        self.head = nn.Linear(self.backbone.config.hidden_size, 1000)

    def forward(self, x):
        out = self.backbone(pixel_values=x)
        pooled = out.pooler_output
        return self.head(pooled)

def get_clip_vision_model(device):
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    def transform(img):
        return processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0)

    return CLIPWithHead(device).to(device).eval(), transform


# Captum

In [10]:
def explain_ig(model, img, label):
    ig = IntegratedGradients(model)
    return ig.attribute(img.unsqueeze(0), target=label)

def explain_saliency(model, img, label):
    sal = Saliency(model)
    return sal.attribute(img.unsqueeze(0), target=label)

def explain_gradientShap(model, img, label):
    gs = GradientShap(model)
    baseline = torch.zeros_like(img)
    return gs.attribute(img.unsqueeze(0), baselines=baseline.unsqueeze(0), target=label)

def explain_occlusion(model, img, label):
    occ = Occlusion(model)
    return occ.attribute(
        img.unsqueeze(0),
        target=label,
        sliding_window_shapes=(3,15,15),
        strides=(3,8,8)
    )


In [11]:
class SHAPWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        if isinstance(x, list):
            x = x[0]
        return self.model(x.float())


# Quantus

In [12]:
def prepare_for_quantus(imgs, attrs):
    # imgs: (B,3,224,224)
    # attrs: (B,C,224,224)

    imgs_small = F.interpolate(
        imgs, size=(56,56), mode="bilinear", align_corners=False
    ).detach().cpu().numpy()

    attrs_small = F.interpolate(
        attrs, size=(56,56), mode="bilinear", align_corners=False
    ).detach().cpu().numpy()

    return imgs_small, attrs_small


In [13]:
import torch.nn.functional as F
import numpy as np

def reduce_attr_for_quantus(attr):
    # attr may be (C,H,W), (1,C,H,W), (B,C,H,W), (B,1,C,H,W)

    if attr.dim() == 3:
        attr = attr.unsqueeze(0)  # (1,C,H,W)

    if attr.dim() == 5:
        attr = attr.squeeze(1)  # (B,C,H,W)

    if attr.dim() != 4:
        raise ValueError(f"Expected (B,C,H,W), got {attr.shape}")

    # Upsample attribution to match model input size
    a = F.interpolate(attr, size=(224,224), mode="bilinear", align_corners=False)

    return a.detach().cpu().numpy()


In [14]:
def evaluate_quantus(model, img, label, attr):
    model_cpu = model.to("cpu")
    model_cpu.eval()

    x_np = img.unsqueeze(0).detach().cpu().numpy()
    a_np = reduce_attr(attr)
    a_np = np.expand_dims(a_np, axis=0)
    y_np = np.array([label])

    metrics = [
        FaithfulnessCorrelation(),
        Robustness(),
        Localisation(),
        Complexity(),
        AxiomaticAttribution(),
        Randomisation()
    ]

    results = {}
    for m in metrics:
        try:
            results[m.__class__.__name__] = m(
                model=model_cpu,
                x_batch=x_np,
                a_batch=a_np,
                y_batch=y_np
            )
        except Exception as e:
            print(f"[Quantus error] {m.__class__.__name__}: {e}")
            results[m.__class__.__name__] = np.nan

    model.to(device)
    return results


In [15]:
def quantus_explain_func(model, inputs, targets, **kwargs):
    model.eval()
    ig = IntegratedGradients(model)

    attributions = []
    for i in range(inputs.shape[0]):
        attr = ig.attribute(
            inputs[i].unsqueeze(0),
            target=int(targets[i])
        )
        attributions.append(attr)

    return torch.cat(attributions, dim=0).detach().cpu().numpy()

In [16]:
def evaluate_quantus_static(model, img, label, attr):
    model_q = model.to("cpu").eval()

    x_np = img.detach().cpu().numpy()
    a_np = reduce_attr_for_quantus(attr)

    # 🔥 important : réduire canal
    a_np = np.mean(a_np, axis=1, keepdims=True)

    y_np = np.array([label])

    metrics = [
        FaithfulnessCorrelation(),
        Sparseness(),
        Complexity()
    ]

    results = {}
    for m in metrics:
        try:
            results[m.__class__.__name__] = m(
                model=model_q,
                x_batch=x_np,
                a_batch=a_np,
                y_batch=y_np
            )
        except Exception as e:
            results[m.__class__.__name__] = np.nan

    return results

In [17]:
def evaluate_quantus_dynamic(model, img, label):
    model_q = model.to("cpu").eval()

    x_np = img.detach().cpu().numpy()
    y_np = np.array([label])

    metrics = [
        MaxSensitivity(),
        AvgSensitivity(),
    ]

    results = {}
    for m in metrics:
        try:
            results[m.__class__.__name__] = m(
                model=model_q,
                x_batch=x_np,
                y_batch=y_np,
                explain_func=quantus_explain_func
            )
        except Exception as e:
            results[m.__class__.__name__] = np.nan

    return results

# Execution

In [18]:
import torch
import pickle
import os

ATTR_DIR = "saved_attributions"
os.makedirs(ATTR_DIR, exist_ok=True)

all_attrs = []

df = pd.read_pickle(os.path.join(DATASETS_DIR, f"df_{CURRENT_DATASET}.pkl"))
IMAGE_PATHS = df["image_path"].tolist()

MAX_IMAGES = 50
IMAGE_PATHS = IMAGE_PATHS[:MAX_IMAGES]

models = {
    "ResNet50": get_resnet50_model(device),
    "MobileNetV3": get_mobilenet_v3_model(device),  
    "ViT": get_vit_model(device),
    "PVT": get_pvt_model(device),
    "CLIP-Vision": get_clip_vision_model(device)
}


for name, (model, transform) in models.items():
    print(f"\n=== Running {name} (Captum only) ===")

    dataset = ImageDataset(IMAGE_PATHS, transform)
    loader = torch.utils.data.DataLoader(dataset, batch_size=1)

    model_attrs = []

    for idx, img in enumerate(loader):
        img = img.squeeze(0).to(device)

        logits = model(img.unsqueeze(0))
        label = logits.argmax(dim=1).item()

        ig_attr  = explain_ig(model, img, label)
        sal_attr = explain_saliency(model, img, label)
        gs_attr  = explain_gradientShap(model, img, label)
        occ_attr = explain_occlusion(model, img, label)

        model_attrs.append({
            "image_idx": idx,
            "label": label,
            "IG": ig_attr.cpu(),
            "Saliency": sal_attr.cpu(),
            "GradientShap": gs_attr.cpu(),
            "Occlusion": occ_attr.cpu(),
        })

    all_attrs.append({"model": name, "attrs": model_attrs})

with open("captum_attributions.pkl", "wb") as f:
    pickle.dump(all_attrs, f)


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/177 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_n


=== Running ResNet50 (Captum only) ===


/home/aysel/tfe/.venv/lib/python3.12/site-packages/captum/attr/_core/saliency.py:129: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  gradient_mask = apply_gradient_requirements(inputs_tuple)



=== Running MobileNetV3 (Captum only) ===

=== Running ViT (Captum only) ===

=== Running PVT (Captum only) ===

=== Running CLIP-Vision (Captum only) ===


In [19]:
quantus_results = []

for entry in all_attrs:
    model_name = entry["model"]
    print(f"\n=== Quantus for {model_name} ===")

    model, transform = models[model_name]
    model.eval()

    model_scores = []

    for item in entry["attrs"]:
        idx = item["image_idx"]
        label = item["label"]

        img = transform(Image.open(IMAGE_PATHS[idx])).to(device)
        img = img.unsqueeze(0)

        attr = item["IG"].to(device)

        scores = evaluate_quantus_static(model, img, label, attr)

        model_scores.append(scores)

    quantus_results.append({
        "model": model_name,
        "scores": model_scores
    })



=== Quantus for ResNet50 ===

=== Quantus for MobileNetV3 ===

=== Quantus for ViT ===

=== Quantus for PVT ===

=== Quantus for CLIP-Vision ===


In [50]:
rows = []

for entry in quantus_results:
    model = entry["model"]
    for s in entry["scores"]:
        for metric, value in s.items():

            # Convert lists or arrays to a single float
            if isinstance(value, (list, np.ndarray)):
                if len(value) == 0:
                    val = np.nan
                else:
                    val = float(np.mean(value))
            else:
                try:
                    val = float(value)
                except:
                    val = np.nan

            rows.append([model, metric, val])


df = pd.DataFrame(rows, columns=["model", "metric", "value"])
summary = df.groupby(["model", "metric"]).mean().reset_index()


summary

,model,metric,value
0,CLIP-Vision,Complexity,10.065792
1,CLIP-Vision,Entropy,10.065292
2,CLIP-Vision,FaithfulnessCorrelation,-0.006538
3,CLIP-Vision,RankCorrelation,0.059594
4,CLIP-Vision,Robustness,NaN
5,CLIP-Vision,Sparseness,0.630031
6,CLIP-Vision,Stability,0.091567
7,MobileNetV3,AttentionAlignment,NaN
8,MobileNetV3,Complexity,10.112149
9,MobileNetV3,Entropy,10.111649


In [21]:
from quantus.metrics.robustness.local_lipschitz_estimate import LocalLipschitzEstimate

def evaluate_quantus_robustness(model, img, label):
    model_q = model.to("cpu").eval()

    x_np = img.detach().cpu().numpy()
    y_np = np.array([label])

    metric = LocalLipschitzEstimate()

    try:
        score = metric(
            model=model_q,
            x_batch=x_np,
            y_batch=y_np
        )
    except Exception:
        score = np.nan

    return score


In [22]:
for entry in quantus_results:
    model_name = entry["model"]
    print(f"\n=== Adding Robustness for {model_name} ===")

    model, transform = models[model_name]
    model.eval()

    for i, item in enumerate(entry["scores"]):
        idx = all_attrs[[m["model"] for m in all_attrs].index(model_name)]["attrs"][i]["image_idx"]
        label = all_attrs[[m["model"] for m in all_attrs].index(model_name)]["attrs"][i]["label"]

        # Load image
        img = transform(Image.open(IMAGE_PATHS[idx])).to(device)
        img = img.unsqueeze(0)

        # Compute robustness
        robustness_score = evaluate_quantus_robustness(model, img, label)

        # Add to existing scores
        entry["scores"][i]["Robustness"] = robustness_score



=== Adding Robustness for ResNet50 ===

=== Adding Robustness for MobileNetV3 ===

=== Adding Robustness for ViT ===

=== Adding Robustness for PVT ===

=== Adding Robustness for CLIP-Vision ===


In [42]:
import numpy as np
from scipy.stats import spearmanr
import torch.nn.functional as F

def metric_rank_correlation(model, img, label, attr, patch=16):
    model_cpu = model.to("cpu").eval()
    img_cpu = img.to("cpu")

    # Attribution IG → numpy
    a = attr.detach().cpu().numpy().squeeze()
    a = np.mean(a, axis=0)  # HxW

    H, W = a.shape
    impacts = []
    attrs_patched = []

    with torch.no_grad():
        base = model_cpu(img_cpu).softmax(dim=1)[0, label].item()

        for i in range(0, H, patch):
            for j in range(0, W, patch):

                # 1) Impact du patch
                img_mod = img_cpu.clone()
                img_mod[:, :, i:i+patch, j:j+patch] = 0
                out = model_cpu(img_mod).softmax(dim=1)[0, label].item()
                impacts.append(base - out)

                # 2) Attribution moyenne dans ce patch
                patch_attr = a[i:i+patch, j:j+patch]
                attrs_patched.append(patch_attr.mean())

    impacts = np.array(impacts)
    attrs_patched = np.array(attrs_patched)

    corr, _ = spearmanr(attrs_patched, impacts)
    return float(corr)


In [43]:
def metric_entropy(attr):
    a = attr.detach().cpu().numpy().squeeze()
    a = np.mean(a, axis=0)
    a = np.abs(a)
    a = a / (a.sum() + 1e-8)
    entropy = -np.sum(a * np.log(a + 1e-8))
    return entropy


In [44]:
import torchvision.transforms as T

augmentations = [
    T.RandomRotation(5),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.RandomHorizontalFlip(p=1.0)
]

def metric_stability(model, img, label, attr_original):
    ig = IntegratedGradients(model)
    sims = []

    for aug in augmentations:
        img_aug = aug(img.squeeze(0)).unsqueeze(0)

        attr_aug = ig.attribute(img_aug, target=label)
        a1 = attr_original.detach().cpu().numpy().flatten()
        a2 = attr_aug.detach().cpu().numpy().flatten()

        # similarité cosinus
        sim = np.dot(a1, a2) / (np.linalg.norm(a1)*np.linalg.norm(a2) + 1e-8)
        sims.append(sim)

    return float(np.mean(sims))


In [45]:
for entry in quantus_results:
    model_name = entry["model"]
    print(f"\n=== Adding new XAI metrics for {model_name} ===")

    model, transform = models[model_name]
    model.eval()

    for i, score_dict in enumerate(entry["scores"]):
        idx = all_attrs[[m["model"] for m in all_attrs].index(model_name)]["attrs"][i]["image_idx"]
        label = all_attrs[[m["model"] for m in all_attrs].index(model_name)]["attrs"][i]["label"]
        attr = all_attrs[[m["model"] for m in all_attrs].index(model_name)]["attrs"][i]["IG"]

        img = transform(Image.open(IMAGE_PATHS[idx])).to(device)
        img = img.unsqueeze(0)

        # Rank correlation
        score_dict["RankCorrelation"] = metric_rank_correlation(model, img, label, attr)

        # Entropy
        score_dict["Entropy"] = metric_entropy(attr)



=== Adding new XAI metrics for ResNet50 ===

=== Adding new XAI metrics for MobileNetV3 ===

=== Adding new XAI metrics for ViT ===

=== Adding new XAI metrics for PVT ===

=== Adding new XAI metrics for CLIP-Vision ===


In [47]:
def metric_stability(model, img, label, attr_original):
    model_cpu = model.to("cpu").eval()
    img_cpu = img.to("cpu")

    ig = IntegratedGradients(model_cpu)

    augmentations = [
        T.RandomRotation(5),
        T.ColorJitter(brightness=0.1, contrast=0.1),
        T.RandomHorizontalFlip(p=1.0)
    ]

    sims = []

    for aug in augmentations:
        img_aug = aug(img_cpu.squeeze(0)).unsqueeze(0)

        attr_aug = ig.attribute(img_aug, target=label)

        a1 = attr_original.detach().cpu().numpy().flatten()
        a2 = attr_aug.detach().cpu().numpy().flatten()

        sim = np.dot(a1, a2) / (np.linalg.norm(a1)*np.linalg.norm(a2) + 1e-8)
        sims.append(sim)

    return float(np.mean(sims))


In [49]:
for entry in quantus_results:
    model_name = entry["model"]
    print(f"\n=== Adding new XAI metrics for {model_name} ===")

    model, transform = models[model_name]
    model.eval()

    for i, score_dict in enumerate(entry["scores"]):
        idx = all_attrs[[m["model"] for m in all_attrs].index(model_name)]["attrs"][i]["image_idx"]
        label = all_attrs[[m["model"] for m in all_attrs].index(model_name)]["attrs"][i]["label"]
        attr = all_attrs[[m["model"] for m in all_attrs].index(model_name)]["attrs"][i]["IG"]

        img = transform(Image.open(IMAGE_PATHS[idx])).to(device)
        img = img.unsqueeze(0)

        # Rank correlation
        score_dict["Stability"] = metric_stability(model, img, label, attr)



=== Adding new XAI metrics for ResNet50 ===

=== Adding new XAI metrics for MobileNetV3 ===

=== Adding new XAI metrics for ViT ===

=== Adding new XAI metrics for PVT ===

=== Adding new XAI metrics for CLIP-Vision ===
